In [1]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math


In [20]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=512,
        temperature=0.0,
        model=deployment
    )

In [22]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AddSubsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CCoT_prompt_example.txt").read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/CoT_prompt_1.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        prompt_q = (
            CoT_prompt_examples +
            '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accurately."},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Answer
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        # === Write to Correct/Incorrect Logs
        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            print("wrong")
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:01<03:31,  1.06s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:02<03:25,  1.04s/it]

Accuracy: 2 / 2 = 100.00%


  2%|▏         | 3/200 [00:02<02:12,  1.49it/s]

...

Accuracy: 173 / 198 = 87.37%


100%|█████████▉| 199/200 [02:43<00:00,  1.01it/s]

Accuracy: 174 / 199 = 87.44%


100%|██████████| 200/200 [02:44<00:00,  1.22it/s]

Accuracy: 175 / 200 = 87.50%


In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/Standard_prompt_1.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accuractly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:00<01:20,  2.47it/s]

Accuracy: 1 / 1 = 100.00%


  2%|▏         | 3/200 [00:02<02:13,  1.47it/s]

Accuracy: 2 / 2 = 100.00%
Accuracy: 3 / 3 = 100.00%


  2%|▏         | 4/200 [00:02<01:36,  2.02it/s]

...

Accuracy: 182 / 198 = 91.92%


100%|█████████▉| 199/200 [01:32<00:00,  3.56it/s]

Accuracy: 183 / 199 = 91.96%


100%|██████████| 200/200 [01:33<00:00,  2.15it/s]

Accuracy: 184 / 200 = 92.00%


In [26]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/outputs/CCoT_prompt.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth answer

        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Finish your response with: the answer is <answer>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly.\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Get Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:04<14:28,  4.36s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:07<12:05,  3.66s/it]

Accuracy: 1 / 2 = 50.00%


  2%|▏         | 3/200 [00:11<12:45,  3.89s/it]

...

Accuracy: 161 / 198 = 81.31%


100%|█████████▉| 199/200 [10:05<00:03,  3.04s/it]

Accuracy: 162 / 199 = 81.41%


100%|██████████| 200/200 [10:08<00:00,  3.04s/it]

Accuracy: 163 / 200 = 81.50%
